In [3]:
import numpy as np
import nltk
from scipy.spatial.distance import cosine
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# 1. Base de datos de documentos (se agregan doc4 y doc5)
documentos = {
    "doc1": "Los modelos de inteligencia artificial aprenden de datos.",
    "doc2": "La inteligencia artificial usa datos para entrenar modelos.",
    "doc3": "El mantenimiento de motores eléctricos requiere herramientas adecuadas.",
    "doc4": "Python es un lenguaje de programación muy popular en ciencia de datos e inteligencia artificial.",
    "doc5": "Los sistemas de aprendizaje automático analizan grandes volúmenes de información."
}

# 2. Inicializar herramientas de NLTK
stemmer = SnowballStemmer("spanish")
vacias = set(stopwords.words("spanish"))

def procesar_texto(texto):
    """Tokeniza, convierte a minúsculas, elimina stopwords/puntuación y aplica stemming."""
    tokens = word_tokenize(texto.lower(), language="spanish")
    return [stemmer.stem(t) for t in tokens if t.isalpha() and t not in vacias]

# 3. Procesar todos los documentos de la base de datos
doc_procesados = {nombre: procesar_texto(texto) for nombre, texto in documentos.items()}

# 4. Solicitar la pregunta/consulta al usuario por teclado
pregunta_usuario = input("Ingresa tu pregunta o búsqueda: ")
pregunta_procesada = procesar_texto(pregunta_usuario)

# 5. Construir el vocabulario global (documentos + pregunta)
vocabulario = sorted(list(set(
    [p for lista in doc_procesados.values() for p in lista] + pregunta_procesada
)))

def vectorizar(lista_tokens):
    """Crea un vector de conteo de palabras (Bolsa de Palabras / Bag of Words)."""
    return np.array([lista_tokens.count(p) for p in vocabulario])

# 6. Vectorizar los documentos y la pregunta del usuario
vectores_doc = {nombre: vectorizar(lista) for nombre, lista in doc_procesados.items()}
vector_pregunta = vectorizar(pregunta_procesada)

# 7. Calcular la similitud del coseno entre la pregunta y cada documento
similitudes = {}
for nombre, vector_doc in vectores_doc.items():
    # Evitar división por cero si la pregunta no coincide con ninguna palabra del vocabulario
    if np.all(vector_doc == 0) or np.all(vector_pregunta == 0):
        similitudes[nombre] = 0.0
    else:
        # Similitud del coseno = 1 - distancia del coseno
        similitudes[nombre] = 1.0 - cosine(vector_pregunta, vector_doc)

# 8. Identificar el documento con mayor similitud
doc_mas_parecido = max(similitudes, key=similitudes.get)
max_similitud = similitudes[doc_mas_parecido]

# 9. Mostrar resultados
print("\n RESULTADOS DE LA BÚSQUEDA ")
print(f"Pregunta ingresada: '{pregunta_usuario}'")
print("\nSimilitud con cada documento:")
for nombre, sim in similitudes.items():
    print(f"  - {nombre}: {sim:.3f}")

print("\n" + "="*50)
if max_similitud > 0:
    print(f" DOCUMENTO MÁS PARECIDO: {doc_mas_parecido} (Similitud: {max_similitud:.3f})")
    print(f" Contenido: \"{documentos[doc_mas_parecido]}\"")
else:
    print(" No se encontró ningún documento parecido a la consulta.")
print("="*50)


 RESULTADOS DE LA BÚSQUEDA 
Pregunta ingresada: '¿Qué modelos usan inteligencia artificial?'

Similitud con cada documento:
  - doc1: 0.671
  - doc2: 0.612
  - doc3: 0.000
  - doc4: 0.354
  - doc5: 0.000

 DOCUMENTO MÁS PARECIDO: doc1 (Similitud: 0.671)
 Contenido: "Los modelos de inteligencia artificial aprenden de datos."
